In [3]:
import pandas as pd

data = pd.read_csv(
    r"C:\Users\leoma\OneDrive\Documents\New folder\iris_demo (1).csv"
)

print(data.head())

print(data.shape)

print(data.isna().sum())

print(
    data["species"].value_counts()
)

   sepal_length_cm  sepal_width_cm  petal_length_cm  petal_width_cm species
0              5.1             3.5              1.4             0.2  setosa
1              4.9             3.0              1.4             0.2  setosa
2              4.7             3.2              1.3             0.2  setosa
3              4.6             3.1              1.5             0.2  setosa
4              5.0             3.6              1.4             0.2  setosa
(150, 5)
sepal_length_cm    0
sepal_width_cm     0
petal_length_cm    0
petal_width_cm     0
species            0
dtype: int64
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X = data.drop(
    columns="species"
)

y = data["species"]

X_develop, X_test, y_develop, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_develop,
    y_develop,
    test_size=0.25,
    stratify=y_develop,
    random_state=42
)

print(
    len(X_train),
    len(X_valid),
    len(X_test)
)

90 30 30


Validation and Test Split

The dataset was divided into training, validation, and test sets.

The test set remains untouched until the final evaluation.

Stratified sampling preserves the species proportions across all data splits.

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(
    X_train,
    y_train
)

baseline_pred = baseline.predict(
    X_valid
)

print(
    "Baseline accuracy:",
    accuracy_score(
        y_valid,
        baseline_pred
    )
)

Baseline accuracy: 0.3333333333333333


Baseline

The baseline classifier predicts the most common species.

The baseline accuracy is approximately one third because there are three species with similar class sizes.

The neural network should perform better than this simple guessing strategy.

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

model = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(8,),
        solver="lbfgs",
        max_iter=1000,
        random_state=42
    )
)

model.fit(
    X_train,
    y_train
)

valid_pred = model.predict(
    X_valid
)

print(
    "Validation accuracy:",
    accuracy_score(
        y_valid,
        valid_pred
    )
)

Validation accuracy: 0.9333333333333333


Neural Network

The network contains one hidden layer with eight neurons.

StandardScaler was used before training.

The neural network should achieve much higher accuracy than the baseline model.

In [8]:
comparison = pd.DataFrame(
    {
        "actual":
            y_valid.to_numpy(),

        "predicted":
            valid_pred
    }
)

print(
    comparison.head()
)

print(
    "Validation errors:",
    (
        comparison.actual
        !=
        comparison.predicted
    ).sum()
)

       actual   predicted
0   virginica   virginica
1  versicolor  versicolor
2  versicolor  versicolor
3      setosa      setosa
4      setosa      setosa
Validation errors: 2


In [9]:
smaller = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(4,),
        solver="lbfgs",
        max_iter=1000,
        random_state=42
    )
)

smaller.fit(
    X_train,
    y_train
)

small_pred = smaller.predict(
    X_valid
)

print(
    "Four-neuron validation accuracy:",
    accuracy_score(
        y_valid,
        small_pred
    )
)

print(
    "Eight-neuron validation accuracy:",
    accuracy_score(
        y_valid,
        valid_pred
    )
)

Four-neuron validation accuracy: 0.9666666666666667
Eight-neuron validation accuracy: 0.9333333333333333


Hidden Layer Comparison

The four-neuron and eight-neuron networks were compared using the same training and validation data.

Only the hidden layer size changed, making the comparison fair.

If the accuracies are equal, the smaller network is preferred because it is simpler.

In [10]:
chosen_size = (
    4
    if accuracy_score(
        y_valid,
        small_pred
    ) >= accuracy_score(
        y_valid,
        valid_pred
    )
    else 8
)

final_model = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(chosen_size,),
        solver="lbfgs",
        max_iter=1000,
        random_state=42
    )
)

final_model.fit(
    X_develop,
    y_develop
)

test_pred = final_model.predict(
    X_test
)

print(
    "Final test accuracy:",
    accuracy_score(
        y_test,
        test_pred
    )
)

new_flower = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=X.columns
)

print(
    "New flower prediction:",
    final_model.predict(
        new_flower
    )[0]
)

Final test accuracy: 0.9333333333333333
New flower prediction: setosa


Final Reflection

A neural network was trained to classify iris flowers using four measurements.

The neural network performed much better than the baseline classifier.

The final model was selected using validation accuracy and then evaluated once on the reserved test set.

The new flower was predicted using the final trained model.

A high test score does not guarantee perfect performance on all future flowers.